# Example 5: Agentic Operations Orchestrator with Human Escalation
Companion notebook for **Example 5** of Chapter 6.

Flow: `monitor -> diagnose -> propose action -> risk check -> approve or execute`. Safe, low-risk actions run directly; risky or unclear ones pause for a human. This notebook uses LangGraph's real **`interrupt()` + checkpointer** pause-and-resume mechanism (introduced in Chapter 4, referenced by name in this chapter's text) rather than simulating approval with a pre-filled dict -- the graph genuinely stops and waits. Self-contained -- run this notebook on its own.

In [1]:
from typing import TypedDict


class IncidentState(TypedDict, total=False):
    incident_id: str
    signal: dict
    diagnosis: str
    proposed_action: dict
    rationale: str
    expected_impact: str
    risk_score: float
    needs_approval: bool
    approval: dict
    execution_result: dict
    status: str

## A small MCP server for the operations tools
Each of these is the kind of high-impact action the chapter warns needs an approval gate before it runs.

In [2]:
%%writefile operations_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("OperationsServer")


@mcp.tool()
def isolate_service(service: str) -> dict:
    """Isolate a service from traffic to contain an incident."""
    return {"ok": True, "service": service, "status": "isolated"}


@mcp.tool()
def restart_service(service: str) -> dict:
    """Restart a service to clear a bad process state."""
    return {"ok": True, "service": service, "status": "restarted"}


@mcp.tool()
def scale_service(service: str, replicas: int) -> dict:
    """Scale a service to a given number of replicas."""
    return {"ok": True, "service": service, "replicas": replicas, "status": "scaled"}


@mcp.tool()
def failover_region(service: str) -> dict:
    """Fail a service over to a redundant region."""
    return {"ok": True, "service": service, "status": "failed_over"}


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing operations_server.py


In [3]:
import sys, os

# Jupyter's stdout/stderr streams don't expose a real OS file descriptor, which crashes
# the MCP SDK's Windows subprocess-spawning fallback (it isn't needed outside Jupyter).
if os.name == "nt":
    try:
        sys.stderr.fileno()
    except Exception:
        sys.stderr = os.fdopen(os.dup(2), "w")

import asyncio
import json
import sys
from pathlib import Path
from typing import Any

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

async def call_tool(session: ClientSession, name: str, arguments: dict) -> Any:
    """Call an MCP tool and unwrap its result into a plain Python value."""
    result = await session.call_tool(name, arguments)
    if result.structuredContent is not None:
        content = result.structuredContent
        return content.get("result", content) if isinstance(content, dict) else content
    text = result.content[0].text
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text

**Monitor and diagnose** -- turn a raw signal into a human-readable diagnosis:

In [4]:
def monitor_incident(state: IncidentState) -> IncidentState:
    return {"status": "monitoring"}


def diagnose_incident(state: IncidentState) -> IncidentState:
    symptom = state["signal"].get("symptom", "").lower()
    service = state["signal"].get("service", "unknown-service")
    if "exfiltration" in symptom or "security" in symptom:
        diagnosis = f"Possible security incident on {service}"
    elif "latency" in symptom or "timeout" in symptom:
        diagnosis = f"Traffic pressure or dependency slowdown on {service}"
    else:
        diagnosis = f"Operational issue detected on {service}"
    return {"diagnosis": diagnosis}

**Propose action** -- picks a remediation and estimates its risk. Present *what* the agent wants to do, *why*, and the *expected impact* before anything runs -- this is what makes the step inspectable and governable:

In [5]:
def propose_action(state: IncidentState) -> IncidentState:
    symptom = state["signal"].get("symptom", "").lower()
    service = state["signal"].get("service", "unknown-service")
    severity = state["signal"].get("severity", "medium")

    if "exfiltration" in symptom or "security" in symptom:
        action = {"tool": "isolate_service", "args": {"service": service}}
        rationale = "Security-like signal needs fast containment."
        expected_impact = "Contains blast radius but may reduce availability."
        risk_score = 0.92
    elif "latency" in symptom or "timeout" in symptom:
        action = {"tool": "scale_service", "args": {"service": service, "replicas": 2}}
        rationale = "Latency looks like load pressure, so scaling is the safest first action."
        expected_impact = "Should reduce latency with low operational risk."
        risk_score = 0.34
    elif severity == "critical":
        action = {"tool": "failover_region", "args": {"service": service}}
        rationale = "Critical incident with broad impact; failover needs review."
        expected_impact = "May restore service via redundancy, but high operational impact."
        risk_score = 0.85
    else:
        action = {"tool": "restart_service", "args": {"service": service}}
        rationale = "Restart can clear a bad process state for application errors."
        expected_impact = "May restore service quickly, with short disruption."
        risk_score = 0.58

    return {"proposed_action": action, "rationale": rationale, "expected_impact": expected_impact, "risk_score": risk_score}

**`decide_execution`** and **`request_approval`** -- exactly as shown in the book. `interrupt()` pauses the graph and hands the review payload out to whoever is waiting on the other end of `invoke()`; the graph won't continue until it's resumed:

In [6]:
def decide_execution(state: IncidentState) -> IncidentState:
    risky = state["risk_score"] > 0.7
    ambiguous = not state.get("proposed_action")
    return {
        "needs_approval": risky or ambiguous,
        "status": "awaiting_approval" if (risky or ambiguous) else "ready_to_execute",
    }


def request_approval(state: IncidentState) -> IncidentState:
    review = {
        "incident_id": state["incident_id"],
        "action": state["proposed_action"],
        "rationale": state["rationale"],
        "expected_impact": state["expected_impact"],
        "risk_score": state["risk_score"],
    }
    decision = interrupt(review)  # pauses here until the graph is resumed with a decision
    return {"approval": decision, "status": "ready_to_execute"}

In [7]:
async def execute_action(state: IncidentState, mcp: ClientSession) -> IncidentState:
    action = state["proposed_action"]
    result = await call_tool(mcp, action["tool"], action["args"])
    return {"execution_result": result, "status": "executed"}


def route_after_decide(state: IncidentState) -> str:
    return "request_approval" if state["needs_approval"] else "execute_action"

## Wiring the graph
Compiled with a `checkpointer` -- required for `interrupt()`/resume to work, since the paused state has to be saved somewhere between the two `invoke()` calls.

In [8]:
def build_graph(mcp_session: ClientSession):
    graph = StateGraph(IncidentState)

    async def execute_action_node(state):
        return await execute_action(state, mcp_session)

    graph.add_node("monitor_incident", monitor_incident)
    graph.add_node("diagnose_incident", diagnose_incident)
    graph.add_node("propose_action", propose_action)
    graph.add_node("decide_execution", decide_execution)
    graph.add_node("request_approval", request_approval)
    graph.add_node("execute_action", execute_action_node)

    graph.add_edge(START, "monitor_incident")
    graph.add_edge("monitor_incident", "diagnose_incident")
    graph.add_edge("diagnose_incident", "propose_action")
    graph.add_edge("propose_action", "decide_execution")
    graph.add_conditional_edges(
        "decide_execution", route_after_decide,
        {"request_approval": "request_approval", "execute_action": "execute_action"},
    )
    graph.add_edge("request_approval", "execute_action")
    graph.add_edge("execute_action", END)
    return graph.compile(checkpointer=MemorySaver())

## Running it: pause, inspect, then resume
The MCP session stays open across both `invoke()` calls below -- in a real deployment the review would be handed to an operator (Slack, a dashboard, ...) and the second `invoke()` would happen later, in a different process, using the same `thread_id`.

In [9]:
server_path = Path.cwd() / "operations_server.py"
params = StdioServerParameters(command=sys.executable, args=[str(server_path)])

async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        app = build_graph(session)

        config = {"configurable": {"thread_id": "INC-9001"}}
        signal = {"service": "payments-api", "severity": "critical", "symptom": "security exfiltration alert"}

        first = await app.ainvoke({"incident_id": "INC-9001", "signal": signal}, config=config)
        print("After first invoke -- status:", first["status"])
        review = first["__interrupt__"][0].value
        print(f"Paused for approval: {review['action']['tool']} on {review['action']['args']['service']}"
              f" (risk={review['risk_score']}) -- {review['rationale']}")

        second = await app.ainvoke(
            Command(resume={"decision": "approved", "reviewer": "oncall-manager"}),
            config=config,
        )
        print("\nAfter resume -- status:", second["status"])
        print("execution_result:", second["execution_result"])

After first invoke -- status: awaiting_approval
Paused for approval: isolate_service on payments-api (risk=0.92) -- Security-like signal needs fast containment.

After resume -- status: executed
execution_result: {'ok': True, 'service': 'payments-api', 'status': 'isolated'}
